# Preprocessing

In this notebook, a series of preprocessing steps are performed on the data. 
Note that some datasets may not require all of these steps, and some could include additional steps that are not listed here and could interfere with the performance of the segmentation tools.

## DIY: Data preparation
The data should be organized in a specific way to be used by the segmentation tools. Otherwise, in order to train models/test the existing ones on new datasets, you should define a new datamodel (see src/datamodels.py).
The following is an example of how the data should be organized:
```
wmh
├── training
│   ├── center_name
│   │   ├── subject_id
│   │   │   ├── pre
│   │   │   │   ├── FLAIR.nii.gz
│   │   │   │   ├── T1.nii.gz
│   │   │   ├── wmh.nii.gz
├── test
│   ├── center_name
│   │   ├── subject_id
│   │   │   ├── pre
│   │   │   │   ├── FLAIR.nii.gz
│   │   │   │   ├── T1.nii.gz
```

Note that even the images are located in a 'pre' folder, they are not preprocessed yet, but they will be in the following steps.

# Bias field correction
The pretrained models were trained on images that were bias-field corrected.

In [ ]:
import os

import SimpleITK as sitk

src_folder = '/home/franco/Code/datasets/wmh/training/UMCL'
dst_folder = '/home/franco/Code/datasets/wmh_prep/training/UMCL'

filename_filter = ['FLAIR.nii.gz', 'T1.nii.gz']

# run the bias field correction preserving the folder structure for all subfolders
for root, dirs, files in os.walk(src_folder):
    out_folder = os.path.join(dst_folder, os.path.relpath(root, src_folder))
    os.makedirs(out_folder, exist_ok=True)
    for filename in files:
        if filename in filename_filter:
            img = sitk.ReadImage(os.path.join(root, filename))
            corr = sitk.N4BiasFieldCorrection(sitk.Cast(img, sitk.sitkFloat32))
            sitk.WriteImage(corr, os.path.join(out_folder, filename))
            print(f'Bias field correction {os.path.join(out_folder, filename)}')

# Take images to the same space
The pretrained models were trained on patches of images with the same spacing and orientation.

In [9]:
from nibabel.processing import conform
import numpy as np
import nibabel as nib

src_folder = '/home/franco/Code/datasets/wmh_prep/training/UMCL'
dst_folder = '/home/franco/Code/datasets/wmh_prep/training/UMCL'

filename_filter = ['FLAIR.nii.gz', 'T1.nii.gz']

# run conform on the wmh_prep images overwriting the original images
for root, dirs, files in os.walk(src_folder):
    for filename in files:
        if filename in filename_filter:
            im_pth = os.path.join(root, filename)
            image = conform(im_pth, out_shape=(240, 240, 155),
                            voxel_size=(1.0, 1.0, 1.0), orientation='RAS')
            data = image.get_fdata()
            data = data.astype(np.int16)

            standarized_image = nib.Nifti1Image(data, affine=np.eye(4))

            nib.save(standarized_image, im_pth)
            print(f'Processed {im_pth}')
            

AttributeError: 'str' object has no attribute 'ndim'